# 03 - Preparação do Modelo (Data Preparation)

Este notebook foca na preparação dos dados para a fase de modelagem, seguindo o framework CRISP-DM. Aqui realizaremos o tratamento de variáveis categóricas, escalonamento de atributos numéricos e a divisão dos dados para treinamento e avaliação.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

# Configurações de visualização
pd.set_option('display.max_columns', None)

## 1. Carregamento dos Dados

Carregamos o dataset processado no EDA.

In [2]:
# Carregar o dataset
data_path = os.path.join('..', 'data', 'processed', 'synthetic_energy_emissions_dataset.csv')
df = pd.read_csv(data_path)

# Derivar colunas de mes e estacao a partir da coluna data
df['data'] = pd.to_datetime(df['data'])
df['mes'] = df['data'].dt.month

def get_estacao(mes):
    if mes in [12, 1, 2]: return 'verao'
    elif mes in [3, 4, 5]: return 'outono'
    elif mes in [6, 7, 8]: return 'inverno'
    else: return 'primavera'

df['estacao'] = df['mes'].apply(get_estacao)

print(f'Formato do dataset: {df.shape}')
df.head()


Formato do dataset: (100000, 12)


,Unnamed: 0,id_empresa,data,estado,setor,porte,tipo_combustivel,consumo_kwh,fonte_energia,emissao_co2,mes,estacao
0,0,C745246,2025-01-12,SP,residencial,small,electric,144.991569,hidrelétrica,2.681433,1,verao
1,1,C700352,2025-06-17,SC,comercial,large,electric,1135.388383,térmica,621.047965,6,inverno
2,2,C652723,2025-05-07,SP,industrial,large,electric,77578.033922,solar,1437.772090,5,outono
3,3,C224352,2025-12-04,PA,industrial,large,electric,35533.629693,hidrelétrica,770.343313,12,verao
4,4,C521541,2025-06-01,MG,rural,large,electric,849.738656,eólica,9.439465,6,inverno


## 2. Seleção de Atributos (Feature Selection)

Definimos nossas variáveis independentes (X) e a variável alvo (y - `co2_emission`).

In [3]:
# Variaveis alvo e atributos
target = 'emissao_co2'
features = ['consumo_kwh', 'setor', 'estado', 'fonte_energia', 'mes', 'estacao']

X = df[features]
y = df[target]

print(f"Variavel alvo: {target}")
print(f"Atributos ({len(features)}): {features}")


Variavel alvo: emissao_co2
Atributos (6): ['consumo_kwh', 'setor', 'estado', 'fonte_energia', 'mes', 'estacao']


## 3. Divisão Treino e Teste

Dividimos os dados em 80% para treinamento e 20% para teste.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamanho Treino: {X_train.shape}")
print(f"Tamanho Teste: {X_test.shape}")

Tamanho Treino: (80000, 6)
Tamanho Teste: (20000, 6)


## 4. Pré-processamento (Pipeline)

Aplicamos as transformações necessárias de forma consistente.

In [5]:
# Identificar colunas numericas e categoricas
numeric_features = ['consumo_kwh', 'mes']
categorical_features = ['setor', 'estado', 'fonte_energia', 'estacao']

# Definir transformadores
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinar transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('[OK] Pipeline de pre-processamento configurado.')


[OK] Pipeline de pre-processamento configurado.
